# **`Inatel - C24 (Inteligência Artificial) - 2026/2`**

# <font color='green'><u>**Atividade 05 - RNN**</u></font>

## <font color='#2D9CDB'>**LEIA ATENTAMENTE AS INSTRUÇÕES A SEGUIR**</font>
- <font color='red'>**A atividade deverá ser resolvida em grupos de até 3 integrantes.**</font>
- Importe este notebook no [Google Colab](https://colab.research.google.com/) para resolver os exercícios;
- Consulte o material disponibilizado pelo(a) professor(a) para revisar os conceitos;
- Utilize os recursos disponíveis na Internet (documentações e artigos científicos) para complementar seus estudos;
- <font color='red'>**A ATIVIDADE DEVERÁ SER REALIZADA _AO VIVO_ COM O [MONITOR](mailto:matheus.botelho@ges.inatel.br), EM UM DOS SEGUINTES HORÁRIOS:**</font>

| Monitor                       | Dia           | Hora                       | Local                                                      |
|-------------------------------|---------------|----------------------------|----------------------------------------------------------- |
| Matheus Botelho Sampaio Netto | <font color='#2D9CDB'>Terça-feira</font>   | <font color='#2D9CDB'>17:30</font> | Teams (ao vivo)       |
| Matheus Botelho Sampaio Netto | <font color='#2D9CDB'>Quarta-feira</font>   | <font color='#2D9CDB'>17:30</font> | Teams (ao vivo)       |
| Matheus Botelho Sampaio Netto | <font color='LightSeaGreen'>Sábado</font>  | <font color='LightSeaGreen'>13:30</font> | Teams (ao vivo) |

- <font color='red'>**NÃO**</font> remova as células de Código já presentes neste notebook;
- <font color='red'>**NÃO**</font> modifique as células de Markdown (em <font color='LightSeaGreen'><u>verde</font> ou <font color='#2D9CDB'><u>azul</u></font>) presentes neste notebook;
- Após cada questão, há uma célula para você implementar e responder a questão;
- É permitido adicionar mais células (de código ou markdown) antes da próxima pergunta;
- Caso precise utilizar bibliotecas que não estão instaladas nativamente no Colab, inclua uma célula de código com o comando de instalação.</font>
  - Exemplo: `!pip install nome_da_biblioteca`

- <font color='magenta'>**A ENTREGA É COMPOSTA POR DOIS ITENS, AMBOS ANEXADOS À TAREFA DO TEAMS:**</font>
  - <font color='red'>**1) O notebook com a resolução**</font>, baixado do [Google Colab](https://colab.research.google.com/) com a saída de todas as células preservada;
  - <font color='red'>**2) Um vídeo de apresentação de até 15 minutos**</font>, no qual o grupo explica as resoluções de forma concisa.
    - <font color='red'>**Todos os integrantes devem participar**</font>, dividindo as questões entre si;
    - Questões de implementação direta podem ser apresentadas em bloco;
    - As questões dissertativas devem ser justificadas individualmente, explicando o porquê de cada escolha e não apenas o que foi feito;
    - Mantenha o notebook visível na tela durante a apresentação.
  - <font color='red'>**A entrega deve ser feita por apenas um integrante da equipe**</font>, em nome de todo o grupo. Os demais não devem submeter a tarefa;
  - <font color='red'>**Identifique todos os integrantes**</font> (nome completo e matrícula) na célula de markdown abaixo.
  - <font color='red'>**NÃO**</font> publique este notebook, sua resolução ou o vídeo em repositórios ou plataformas públicas, nem os compartilhe com outros grupos;
      - Se desejar versionar o trabalho, utilize um repositório <font color='red'>**privado**.

- <font color='magenta'>**SOBRE O USO DE FERRAMENTAS DE IA GENERATIVA:**</font>
  - O uso é <font color='red'>**permitido**</font> como apoio ao estudo, para esclarecer conceitos, consultar documentação e revisar código;
  - Se utilizar IA em alguma questão, registre em uma célula de markdown <font color='red'>**quais questões**</font>, qual ferramenta foi usada e para quê.
      - Declarações genéricas, como "usei IA na atividade toda", não atendem a este requisito;
  - <font color='red'>**O uso identificado de IA sem o devido apontamento acarretará a desconsideração das questões envolvidas.**</font>
      - A declaração não reduz a nota; a omissão, sim;
  - Declarar o uso de IA não isenta o grupo de compreender e justificar a resolução;
  - <font color='red'>**No vídeo, o grupo deve justificar as escolhas feitas, e não apenas narrar o código.**</font>
      - Sempre que uma questão admitir mais de um caminho, explique por que o grupo optou por aquele;
  - Todos os integrantes devem falar, e cada um deve comentar ao menos uma questão;
  - Respostas que o grupo não conseguir justificar serão desconsideradas na correção.

## <font color='#2D9CDB'>**INTEGRANTES DO GRUPO**</font>

| Nome                          | Matrícula | E-mail                  |
|-------------------------------|-----------|-------------------------|
| Nome 1                        | XXX       | email_1@curso.inatel.br |
| Nome 2                        | YYY       | email_2@curso.inatel.br |
| Nome 3                        | ZZZ       | email_3@curso.inatel.br |

## <font color='LightSeaGreen'><u>**Preparação**</u></font>

### <font color='LightSeaGreen'>Não altere o conteúdo das duas células a seguir.</font>

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn ucimlrepo tensorflow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import tensorflow as tf
import os
import time

from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)
tf.random.set_seed(42)
tf.keras.utils.set_random_seed(42)

# Se esta configuração gerar erro na GPU do Colab,
# comente a linha a seguir e reexecute o Notebook
tf.config.experimental.enable_op_determinism()

# Para medir o tempo de cada época de treinamento em segundos
# Exemplo de uso: https://www.tensorflow.org/guide/keras/writing_your_own_callbacks
class EpochTimer(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []
    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.perf_counter() - self._t0)

## <font color='LightSeaGreen'><u>**Parte 1 - Conjunto de Dados**</u></font>

### <font color='LightSeaGreen'>Não altere o conteúdo da célula a seguir.</font>

In [ ]:
zip_path = tf.keras.utils.get_file(
    fname="jena_climate_2009_2016.csv.zip",
    origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip",
    cache_dir=os.getcwd(),
    extract=True
)

csv_path = os.path.join(os.path.splitext(zip_path)[0], 'jena_climate_2009_2016.csv')

target_col = "T (degC)"

temperature = pd.read_csv(csv_path)[target_col].values.reshape(-1, 1)

temperature

### <font color='#2D9CDB'>Q1) O dataset [Jena Climate](https://www.kaggle.com/datasets/mnassrib/jena-climate) é amplamente utilizado no estudo de redes neurais recorrentes (RNNs), LSTMs e previsão de séries temporais. Pesquise brevemente sobre o dataset e descreva, em um único parágrafo, o domínio de aplicação da base de dados, a origem dos dados e como passaram a ser usados em aprendizado de máquina, o período de coleta, a frequência de amostragem, os tipos de variáveis meteorológicas registradas e por que esse conjunto de dados é adequado para problemas de previsão temporal utilizando redes neurais recorrentes.</font>

### <font color='#2D9CDB'>Q2) Utilizando a variável <code>temperature</code>, plote a série temporal completa de temperaturas registradas no dataset. Em seguida, informe a quantidade total de amostras, a temperatura mínima, a temperatura máxima, a temperatura média e o desvio padrão da série. Com base no gráfico obtido, descreva brevemente os padrões observados, comentando sobre possíveis tendências, sazonalidades e variações ao longo do tempo.</font>

### <font color='#2D9CDB'>Q3) Divida a série temporal em três subconjuntos: treinamento (60%), validação (20%) e teste (20%), preservando a ordem temporal das amostras. Informe a quantidade de amostras em cada subconjunto e explique por que, em problemas de previsão temporal, não é adequado embaralhar os dados antes da divisão.</font>

### <font color='#2D9CDB'>Q4) Normalize os conjuntos de treinamento, validação e teste utilizando a técnica Min-Max Scaling, ajustando o scaler apenas no conjunto de treinamento. Apresente os valores mínimo e máximo de cada conjunto antes e depois da normalização. Explique brevemente por que a normalização é importante para o treinamento de redes neurais e por que o scaler deve ser ajustado apenas no conjunto de treinamento.</font>

# <font color='LightSeaGreen'><u><b>Parte 2 - Preparação das Séries Temporais</b></u></font>

### <font color='#2D9CDB'>Q5) Para treinar uma LSTM, é necessário organizar a série temporal em sequências de entrada e saídas desejadas. Implemente uma função chamada <code>create_sequences()</code> que utilize uma janela temporal de 30 amostras, de forma que cada entrada seja composta pelas 30 temperaturas anteriores e a saída corresponda à temperatura seguinte. Gere os conjuntos de treinamento, validação e teste e informe as dimensões de <code>X_train</code> e <code>y_train</code>. Explique brevemente o significado de cada dimensão e a quanto tempo de histórico a janela corresponde.</font>

### <font color='#2D9CDB'>Q6) Exiba graficamente a primeira sequência presente em <code>X_train</code> e destaque o valor correspondente em <code>y_train</code>. Explique como essa sequência será utilizada pela LSTM para realizar a previsão da próxima temperatura.</font>

# <font color='LightSeaGreen'><u><b>Parte 3 - Construindo a Primeira LSTM</b></u></font>

### <font color='#2D9CDB'>Q7) Construa uma rede neural recorrente composta por uma camada LSTM com uma unidade de memória e uma camada de saída totalmente conectada com um neurônio. Utilize a função de perda Erro Quadrático Médio (MSE - <code>mean_squared_error</code>) e o otimizador Adam. Apresente o resumo da arquitetura gerado pelo método <code>summary()</code>, informe o número total de parâmetros treináveis do modelo e mostre como esse número é obtido.</font>

### <font color='#2D9CDB'>Q8) Treine o modelo por 10 épocas utilizando os conjuntos de treinamento e validação. Em seguida, plote as curvas de perda de treinamento e validação ao longo das épocas. Com base nos gráficos obtidos, descreva brevemente o comportamento do treinamento e indique se há evidências de sobreajuste (overfitting) ou problemas de generalização.</font>

### <font color='#2D9CDB'>Q9) Avalie o modelo utilizando o conjunto de teste e reporte o erro quadrático médio (MSE) e a raiz do erro quadrático médio (RMSE) em graus Celsius. Compare o resultado com uma previsão ingênua (baseline de persistência), que usa a última temperatura da janela como previsão. Explique brevemente o que essas métricas representam no contexto do problema e se o modelo supera o baseline.</font>

### <font color='#2D9CDB'>Q10) Utilize o modelo treinado para gerar previsões no conjunto de teste. Plote em um mesmo gráfico, em graus Celsius, os valores reais, os valores previstos e a previsão do baseline de persistência para as 100 primeiras amostras do conjunto de teste. Descreva brevemente a qualidade das previsões obtidas, comentando se há atraso entre as curvas real e prevista.</font>

# <font color='LightSeaGreen'><u><b>Parte 4 - Comparação de Arquiteturas</b></u></font>

### <font color='#2D9CDB'>Q11) Modifique a arquitetura da rede para utilizar 10 unidades na camada LSTM. Apresente o resumo da arquitetura gerado pelo método <code>summary()</code> e informe o número total de parâmetros treináveis do modelo. Em seguida, treine a rede por 10 épocas, avalie o modelo no conjunto de teste e reporte o valor de MSE obtido.</font>

### <font color='#2D9CDB'>Q12) Repita o experimento utilizando 20 unidades na camada LSTM. Apresente o resumo da arquitetura gerado pelo método <code>summary()</code> e informe o número total de parâmetros treináveis do modelo. Em seguida, treine a rede por 10 épocas, avalie o modelo no conjunto de teste e reporte o valor de MSE obtido.</font>

### <font color='#2D9CDB'>Q13) Compare os modelos com 1, 10 e 20 unidades na camada LSTM em termos de número de parâmetros treináveis, desempenho no conjunto de teste e tempo de treinamento. Organize os resultados em uma tabela contendo a quantidade de unidades LSTM, o número de parâmetros treináveis, o MSE, o RMSE em °C e o tempo médio por época, incluindo o baseline de persistência como referência. Com base nos resultados obtidos, discuta brevemente a relação entre a complexidade do modelo e sua capacidade preditiva.</font>

# <font color='LightSeaGreen'><u><b>Parte 5 - Influência da Janela Temporal</b></u></font>

### <font color='#2D9CDB'>Q14) Repita o experimento utilizando uma janela temporal de 15 amostras e uma camada LSTM com 10 unidades. Treine o modelo por 10 épocas, avalie o modelo no conjunto de teste e reporte o valor de MSE obtido.</font>

### <font color='#2D9CDB'>Q15) Repita o experimento utilizando uma janela temporal de 60 amostras e uma camada LSTM com 10 unidades. Treine o modelo por 10 épocas, avalie o modelo no conjunto de teste e reporte o valor de MSE obtido.</font>

### <font color='#2D9CDB'>Q16) Compare os resultados obtidos para janelas temporais de 15, 30 e 60 amostras utilizando uma camada LSTM com 10 unidades. Organize os resultados em uma tabela contendo o tamanho da janela, as horas de histórico correspondentes, o MSE, o RMSE em °C e o tempo médio de treinamento por época medido durante o treino. Com base nos resultados, discuta brevemente o impacto do tamanho da janela temporal no desempenho e no custo computacional do modelo.</font>

# <font color='LightSeaGreen'><u><b>Parte 6 - Análise Final</b></u></font>

### <font color='#2D9CDB'>Q17) Com base em todos os experimentos realizados nesta atividade, discuta os fatores que mais influenciaram o desempenho do modelo. Considere o impacto da quantidade de unidades LSTM, do tamanho da janela temporal, do custo computacional e a comparação com o baseline de persistência. Em sua opinião, qual configuração apresentou o melhor equilíbrio entre precisão e eficiência? Justifique sua resposta.</font>